# Módulo 02 — Generación de embeddings semánticos

## Objetivo

Este módulo transforma cada documento multilingüe en un embedding: un vector numérico que representa relaciones semánticas aprendidas previamente por un modelo de lenguaje. TechMind utiliza `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, sin entrenamiento adicional ni fine-tuning.

## Entrada y representación

La entrada es `models/dataset_procesado.json`. Para cada documento se concatena `titulo + texto`, porque el título resume el tema y el cuerpo aporta contexto. El resultado es una lista de textos que conserva las señales lingüísticas en inglés, español y portugués.

## Parámetros importantes

El modelo produce vectores de 384 dimensiones; esa dimensión es una característica de la arquitectura preentrenada. `normalize_embeddings=True` normaliza cada vector para que la comparación por coseno se centre en su orientación. `float32` reduce el consumo de memoria y `batch_size=32` procesa los documentos en lotes para equilibrar velocidad y memoria.

## Flujo

```text
dataset procesado → título + texto → modelo preentrenado → matriz (900, 384)
```

## Salida y relación con el siguiente módulo

La salida es `models/embeddings.npy`, con tipo `float32`. El Módulo 03 utilizará exactamente el mismo orden de filas para construir el índice de vecinos.

## Limitaciones

Los embeddings no son etiquetas ni predicciones de clases. Su calidad depende del modelo preentrenado, del idioma, de la longitud del texto y de la calidad de la limpieza anterior. La dimensión 384 no significa que existan 384 categorías.

In [1]:
import sys
from pathlib import Path
import numpy as np
sys.path.insert(0, str(Path.cwd() / 'src'))
from techmind.config import get_paths, MODEL_NAME
from techmind.persistence.artifacts import load_documents
from techmind.embeddings.encoder import encode_texts, semantic_texts
paths = get_paths(); documents = load_documents(paths.dataset)
embeddings = encode_texts(semantic_texts(documents), model_name=MODEL_NAME, batch_size=32)
np.save(paths.embeddings, embeddings.astype(np.float32))
print(f'[TechMind] Embedding shape: {embeddings.shape}')
assert embeddings.shape == (len(documents), 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[TechMind] Embedding shape: (900, 384)
